### Import

In [2]:
import os
import gc
import time
import pickle 
import itertools
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm.auto import tqdm
import matplotlib.ticker as ticker
from matplotlib import pyplot as plt 
import pyarrow as pa
import pyarrow.parquet as pq
from pyarrow.parquet import ParquetFile
from sklearn.preprocessing import OneHotEncoder
pd.set_option('display.max_columns', 500)

### General parameters

In [3]:
path_data_mimiciii = "./mimiciii/Data/EHR/"
path_data_mimiciv  = "./mimiciv/Data/EHR/"
path_data_covid    = "./covid/Data/EHR/"

In [4]:
race_path_mimiciii = "Extraction/MIMICIII/Data/csvExtract/"
race_path_mimiciv  = "Extraction/MIMICIV/Data/csvExtract/"
race_path_covid    = "Extraction/MIMICIV(V3)/Data/csvExtract/"

### Reading Data

In [5]:
df_mimiciii = pd.read_csv(path_data_mimiciii + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_mimiciii.head(2)

In [6]:
print(df_mimiciii.ICUSTAY_ID.nunique())
print(df_mimiciii.shape)

59706
(1348077, 564)


In [7]:
df_mimiciv = pd.read_csv(path_data_mimiciv + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_mimiciv.head(2)

In [8]:
print(df_mimiciv.stay_id.nunique())
print(df_mimiciv.shape)

71432
(1635041, 545)


In [9]:
df_covid = pd.read_csv(path_data_covid + '0h_to_24h_data.csv', low_memory=False, index_col=False)
df_covid.head(2)

In [10]:
print(df_covid.stay_id.nunique())
print(df_covid.shape)

10561
(240003, 545)


### Drop Repeated Rows

In [11]:
all_columns = list(df_mimiciii.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]
text_columns = ['Note', 'Discharge_Note', 'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_mimiciii.drop(remove_columns, axis=1, inplace=True)
df_mimiciii = df_mimiciii.drop_duplicates()

print(df_mimiciii.ICUSTAY_ID.nunique())
print(df_mimiciii.shape)

59706
(1318008, 302)


In [12]:
all_columns = list(df_mimiciv.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]
text_columns = ['cxr_image', 'cxr_lung', 'cxr_note', 'radiology_note', 'discharge_note', 
                'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_mimiciv.drop(remove_columns, axis=1, inplace=True)
df_mimiciv = df_mimiciv.drop_duplicates()

print(df_mimiciv.stay_id.nunique())
print(df_mimiciv.shape)

71432
(1630475, 290)


In [13]:
all_columns = list(df_covid.columns)
remove_columns = [col for col in all_columns if ('_tslm' in col) or ('_diff' in col)]
text_columns = ['cxr_image', 'cxr_lung', 'cxr_note', 'radiology_note', 'discharge_note', 
                'Heart Rhythm', 'Ventilator Mode', 'Ventilator Type', 'Ventilator']

remove_columns.extend(text_columns)
df_covid.drop(remove_columns, axis=1, inplace=True)
df_covid = df_covid.drop_duplicates()

print(df_covid.stay_id.nunique())
print(df_covid.shape)

10561
(240003, 290)


### Fix Age

In [14]:
df_mimiciii.loc[df_mimiciii['AGE'] >= 95, 'AGE'] = 95
df_mimiciii = df_mimiciii[df_mimiciii.AGE > 16]

In [15]:
df_mimiciv.loc[df_mimiciv['age'] >= 95, 'age'] = 95
df_mimiciv = df_mimiciv[df_mimiciv.age > 16]

In [16]:
df_covid.loc[df_covid['age'] >= 95, 'age'] = 95
df_covid = df_covid[df_covid.age > 16]

### Take first 24 hour LoS

In [17]:
max_rows = df_mimiciii.groupby('ICUSTAY_ID').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [18]:
max_rows = df_mimiciv.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [19]:
max_rows = df_covid.groupby('stay_id').count()
observation_window = max_rows.Bins.max()
print(observation_window)

24


In [20]:
df_mimiciii = df_mimiciii.groupby('ICUSTAY_ID').head(observation_window).reset_index(drop=True)
df_mimiciv  = df_mimiciv.groupby('stay_id').head(observation_window).reset_index(drop=True)
df_covid    = df_covid.groupby('stay_id').head(observation_window).reset_index(drop=True)

### Find Similar Variables

In [21]:
mimiciii_columns = ['ICUSTAY_ID', 'Bins', 'AGE', 'GENDER', 'ETHNICITY', 'Weight', 'Height',
                    
                    'Heart Rate', 'SpO2', 'SvO2', 'Oxygen Saturation', 'Respiratory Rate', 
                    'Respiratory Rate (Total)', 'Respiratory Rate (Set)', 'Temperature', 
                    'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
                    'Non Invasive Blood Pressure diastolic', 'Arterial Blood Pressure mean', 
                    'Arterial Blood Pressure systolic', 'Arterial Blood Pressure diastolic', 
                    
                    'Glucose', 'Creatinine', 'Base Excess', 'BUN', 'Anion Gap',
                    'Bicarbonate', 'Lactate', 'Lactate Dehydrogenase (LD)', 'Hemoglobin', 'Hematocrit', 'pH', 
                    'Bilirubin, Direct', 'Bilirubin, Indirect', 'Bilirubin, Total', 'pO2', 'pCO2', 'PT', 'PTT', 
                    'INR(PT)', 'AST', 'ALT', 'WBC', 'RBC', 'RDW', 'White Blood Cells', 'Red Blood Cells',
                    'Platelet Count', 'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium',
                    'Phosphate', 'Alkaline Phosphatase', 'Ionized Calcium', 'Calcium non-ionized', 'Calcium, Total',
                    'Protein', 'Total Protein', 'Cholesterol, Total', 'Differential-Bands', 'Differential-Monos', 
                    'Differential-Lymphs', 'Differential-Eos', 'Differential-Basos', 'Differential-Neuts', 'Oxygen',
                    'O2 Flow', 'Total CO2', 'Flow Rate (L/min)', 'Mean Airway Pressure',
                    'Plateau Pressure', 'CVP', 'Amylase', 'Albumin', 'Fibrinogen', 'Triglycerides',
                    'Transferrin', 'Ferritin', 'Troponin T', 'Vancomycin (Trough)', 'Vancomycin (Random)',
                    'C-Reactive Protein (CRP)', 'FiO2', 'PEEP', 'PEEP (Set)', 'Tidal Volume',
                    'Tidal Volume (Set)', 'Ventilation Rate', 
                    
                    'UrineOutput_IO', 'Mg-nonIV_IO', 'Insulin_IO', 'K-IV_IO', 'Propofol_IO', 'Heparin_IO', 
                    'Vasopressors_IO', 'Fentanyl_IO', 'Crystalloids_IO','Vancomycin_IO','P.O._IO', 'Dextrose_IO',
                    
                    'Skin Temperature', 'GCS Total', 'GCS - Eye Opening',  'GCS - Verbal Response',  
                    'GCS - Motor Response' , 'Richmond-RAS Scale', 'Goal Richmond-RAS Scale', 'Pain Level', 
                    'Microbio Test',
                    
                    'Pain Present', 'Mental status', 'Risk for Falls', 'Delirium assessment', 'CAM-ICU MS Change',
                    'CAM-ICU Inattention', 'CAM-ICU Disorganized thinking', 'CAM-ICU RASS LOC',  
                    'CAM-ICU Altered LOC', 'Intubated', 'Blood Culture',
                    
                    'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                    
                    'Norepinephrine_PRC', 'Fentanyl_PRC', 'Antibiotic_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                    'Propofol_PRC', 'Phenylephrine_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 'Pantoprazole_PRC',
                    
                    'ICU_EXPIRE_FLAG'] 

In [22]:
mimiciv_columns = ['stay_id', 'Bins', 'age', 'gender', 'race', 'Weight', 'Height',
                   
                   'Heart Rate', 'SpO2', 'SvO2', 'Oxygen Saturation', 'Respiratory Rate', 
                   'Respiratory Rate (Total)', 'Respiratory Rate (Set)', 'Temperature', 
                   'Non Invasive Blood Pressure mean', 'Non Invasive Blood Pressure systolic', 
                   'Non Invasive Blood Pressure diastolic', 'Arterial Blood Pressure mean', 
                   'Arterial Blood Pressure systolic', 'Arterial Blood Pressure diastolic',
                   
                   'Glucose', 'Creatinine', 'Base Excess', 'BUN', 'Anion Gap', 
                   'Bicarbonate',  'Lactate', 'Lactate Dehydrogenase(LDH)', 'Hemoglobin', 'Hematocrit', 'pH', 
                   'Bilirubin, Direct', 'Bilirubin, Indirect', 'Bilirubin, Total', 'pO2', 'pCO2', 'PT', 'PTT', 
                   'INR(PT)', 'AST', 'ALT', 'WBC', 'RBC', 'RDW', 'White Blood Cells', 'Red Blood Cells', 
                   'Platelet Count', 'MCV', 'MCH', 'MCHC', 'Potassium', 'Sodium', 'Chloride', 'Magnesium', 
                   'Phosphate', 'Alkaline Phosphate', 'Ionized Calcium', 'Calcium non-ionized', 'Calcium, Total', 
                   'Protein', 'Total Protein', 'Cholesterol, Total', 'Differential-Bands', 'Differential-Monos', 
                   'Differential-Lymphs', 'Differential-Eos', 'Differential-Basos', 'Differential-Neuts', 'Oxygen', 
                   'O2 Flow', 'Total CO2', 'Flow Rate (L/min)', 'Mean Airway Pressure', 
                   'Plateau Pressure', 'Central Venous Pressure', 'Amylase', 'Albumin', 'Fibrinogen', 'Triglyceride',
                   'Transferrin', 'Ferritin', 'Troponin T', 'Vancomycin (Trough)', 'Vancomycin (Random)', 
                   'C-Reactive Protein (CRP)', 'FiO2', 'PEEP',  'PEEP (Set)', 'Tidal Volume',
                   'Tidal Volume (Set)', 'Ventilation Rate', 
                   
                   'UrineOutput_IO', 'Mg-nonIV_IO', 'Insulin_IO', 'K-IV_IO', 'Propofol_IO', 'Heparin_IO', 
                   'Vasopressors_IO', 'Fentanyl_IO', 'Crystalloids_IO', 'Vancomycin_IO', 'P.O._IO', 'Dextrose_IO',
                   
                   'Skin Temperature', 'Total GCS', 'GCS - Eye Opening', 'GCS - Verbal Response', 
                   'GCS - Motor Response', 'Richmond-RAS Scale', 'Goal Richmond-RAS Scale', 'Pain Level',
                   'Microbio Test',
                   
                   'Pain Present', 'Mental status', 'Risk for Falls', 'Delirium assessment', 'CAM-ICU MS Change', 
                   'CAM-ICU Inattention', 'CAM-ICU Disorganized thinking', 'CAM-ICU RASS LOC', 
                   'CAM-ICU Altered LOC', 'Intubated', 'Blood Culture',
                   
                   'PaO2/FiO2', 'SIRS', 'Shock_Index', 'SOFA', 'SAPSII', 'OASIS',
                   
                   'Norepinephrine_PRC', 'Fentanyl_PRC', 'Antibiotic_PRC', 'Heparin_PRC', 'Insulin_PRC', 
                   'Propofol_PRC', 'Phenylephrine_PRC', 'Vasopressin_PRC', 'Epinephrine_PRC', 'Pantoprazole_PRC',
                   
                   'icu_expire_flag']

In [25]:
mimiciii_columns_all = []

for i in mimiciii_columns:

    if i in list(df_mimiciii.columns):
        mimiciii_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_mimiciii.columns):
        mimiciii_columns_all.append(temp_col)

In [26]:
mimiciv_columns_all = []

for i in mimiciv_columns:

    if i in list(df_mimiciv.columns):
        mimiciv_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_mimiciv.columns):
        mimiciv_columns_all.append(temp_col)

In [27]:
covid_columns_all = []

for i in mimiciv_columns:

    if i in list(df_covid.columns):
        covid_columns_all.append(i)

    temp_col = i + '_ind'
    if temp_col in list(df_covid.columns):
        covid_columns_all.append(temp_col)

### Select Variables

In [31]:
df_mimiciii = df_mimiciii[mimiciii_columns_all]
df_mimiciv  = df_mimiciv[mimiciv_columns_all]
df_covid    = df_covid[covid_columns_all]

In [32]:
wbc_condition_iii = (df_mimiciii['White Blood Cells'].isnull()) & (df_mimiciii['WBC'].notnull())
df_mimiciii.loc[wbc_condition_iii, 'White Blood Cells'] = df_mimiciii['WBC']
df_mimiciii.loc[wbc_condition_iii, 'White Blood Cells_ind'] = df_mimiciii['WBC_ind']

df_mimiciii.drop(columns=['WBC', 'WBC_ind'], inplace=True)

In [33]:
wbc_condition_iv = (df_mimiciv['White Blood Cells'].isnull()) & (df_mimiciv['WBC'].notnull())
df_mimiciv.loc[wbc_condition_iv, 'White Blood Cells'] = df_mimiciv['WBC']
df_mimiciv.loc[wbc_condition_iv, 'White Blood Cells_ind'] = df_mimiciv['WBC_ind']

df_mimiciv.drop(columns=['WBC', 'WBC_ind'], inplace=True)

In [34]:
wbc_condition_iv = (df_covid['White Blood Cells'].isnull()) & (df_covid['WBC'].notnull())
df_covid.loc[wbc_condition_iv, 'White Blood Cells'] = df_covid['WBC']
df_covid.loc[wbc_condition_iv, 'White Blood Cells_ind'] = df_covid['WBC_ind']

df_covid.drop(columns=['WBC', 'WBC_ind'], inplace=True)

### Unifying Column Names

In [37]:
df_mimiciii.columns = list(df_mimiciv.columns)

### Fixing Race

In [42]:
with open(race_path_mimiciii + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary_mimiciii = pickle.load(f)

In [43]:
with open(race_path_mimiciv + 'race_dictionary.pkl', 'rb') as f:
    race_dictionary_mimiciv = pickle.load(f)

In [44]:
general_ethnicity_mapping_mimiciii = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNABLE TO OBTAIN': 'Other/Unknown',
    'UNKNOWN/NOT SPECIFIED': 'Other/Unknown',
    'PATIENT DECLINED TO ANSWER': 'Other/Unknown',
    'OTHER': 'Other/Unknown',
    'MIDDLE EASTERN': 'Other/Unknown',
    'CARIBBEAN ISLAND': 'Other/Unknown',
    'MULTI RACE ETHNICITY': 'Other/Unknown',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other/Unknown',
    
    'ASIAN': 'Asian',
    'ASIAN - THAI': 'Asian',
    'ASIAN - OTHER': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - FILIPINO': 'Asian',
    'ASIAN - JAPANESE': 'Asian',
    'ASIAN - CAMBODIAN': 'Asian',
    'ASIAN - VIETNAMESE': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'AMERICAN INDIAN/ALASKA NATIVE': 'Native American',
    'AMERICAN INDIAN/ALASKA NATIVE FEDERALLY RECOGNIZED TRIBE': 'Native American',
    
    'BLACK/AFRICAN': 'African American',
    'BLACK/HAITIAN': 'African American',
    'BLACK/CAPE VERDEAN': 'African American',
    'BLACK/AFRICAN AMERICAN': 'African American',
    
    'SOUTH AMERICAN': 'Hispanic',
    'HISPANIC OR LATINO': 'Hispanic',
    'HISPANIC/LATINO - CUBAN': 'Hispanic',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic',
    'HISPANIC/LATINO - COLOMBIAN': 'Hispanic',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic',
    'HISPANIC/LATINO - CENTRAL AMERICAN (OTHER)': 'Hispanic'}

In [45]:
general_ethnicity_mapping_mimiciv = {
    
    'WHITE': 'White',
    'WHITE - RUSSIAN': 'White',
    'WHITE - BRAZILIAN': 'White',
    'WHITE - OTHER EUROPEAN': 'White',
    'WHITE - EASTERN EUROPEAN': 'White',
    'PORTUGUESE': 'White',
    
    'UNKNOWN': 'Other/Unknown',
    'UNABLE TO OBTAIN': 'Other/Unknown',
    'PATIENT DECLINED TO ANSWER': 'Other/Unknown',
    'OTHER': 'Other/Unknown',
    'MIDDLE EASTERN': 'Other/Unknown',
    'MULTIPLE RACE/ETHNICITY': 'Other/Unknown',
    'NATIVE HAWAIIAN OR OTHER PACIFIC ISLANDER': 'Other/Unknown',
    'AMERICAN INDIAN/ALASKA NATIVE': 'Other/Unknown',
    
    'ASIAN': 'Asian',
    'ASIAN - CHINESE': 'Asian',
    'ASIAN - SOUTH EAST ASIAN': 'Asian',
    'ASIAN - KOREAN': 'Asian',
    'ASIAN - ASIAN INDIAN': 'Asian',
    
    'BLACK/AFRICAN': 'African American',
    'BLACK/CAPE VERDEAN': 'African American',
    'BLACK/AFRICAN AMERICAN': 'African American',
    'BLACK/CARIBBEAN ISLAND': 'African American',
    
    'SOUTH AMERICAN': 'Hispanic',
    'HISPANIC OR LATINO': 'Hispanic',
    'HISPANIC/LATINO - CUBAN': 'Hispanic',
    'HISPANIC/LATINO - MEXICAN': 'Hispanic',
    'HISPANIC/LATINO - HONDURAN': 'Hispanic',
    'HISPANIC/LATINO - COLUMBIAN': 'Hispanic',
    'HISPANIC/LATINO - DOMINICAN': 'Hispanic',
    'HISPANIC/LATINO - SALVADORAN': 'Hispanic',
    'HISPANIC/LATINO - GUATEMALAN': 'Hispanic',
    'HISPANIC/LATINO - PUERTO RICAN': 'Hispanic',
    'HISPANIC/LATINO - CENTRAL AMERICAN': 'Hispanic'}

In [46]:
def replace_ethnicity_with_names(df, ethnicity_dict):
    
    inv_ethnicity_dict = {v: k for k, v in ethnicity_dict.items()}
    df['race'] = df['race'].map(inv_ethnicity_dict)
    
    return df

In [47]:
def categorize_ethnicity(df, new_mapping):
    
    df['race'] = df['race'].map(new_mapping)
    
    return df

In [48]:
df_mimiciii = replace_ethnicity_with_names(df_mimiciii, race_dictionary_mimiciii)
df_mimiciii = categorize_ethnicity(df_mimiciii, general_ethnicity_mapping_mimiciii)

df_mimiciv = replace_ethnicity_with_names(df_mimiciv, race_dictionary_mimiciv)
df_mimiciv = categorize_ethnicity(df_mimiciv, general_ethnicity_mapping_mimiciv)

df_covid = replace_ethnicity_with_names(df_covid, race_dictionary_mimiciv)
df_covid = categorize_ethnicity(df_covid, general_ethnicity_mapping_mimiciv)

### Encode using 3 bits

In [51]:
all_races = sorted(set(df_mimiciii['race']).union(set(df_mimiciv['race'])).union(set(df_covid['race'])))
race_to_bits = {race: format(i, '03b') for i, race in enumerate(all_races)}

def encode_race(race):
    return list(map(int, race_to_bits[race]))

df_mimiciii_encoded = df_mimiciii['race'].apply(encode_race).apply(pd.Series)
df_mimiciv_encoded  = df_mimiciv['race'].apply(encode_race).apply(pd.Series)
df_covid_encoded    = df_covid['race'].apply(encode_race).apply(pd.Series)

df_mimiciii_encoded.columns = [f'race_bit_{i}' for i in range(3)]
df_mimiciv_encoded.columns  = [f'race_bit_{i}' for i in range(3)]
df_covid_encoded.columns    = [f'race_bit_{i}' for i in range(3)]

df_mimiciii_final = pd.concat([df_mimiciii, df_mimiciii_encoded], axis=1)
df_mimiciv_final  = pd.concat([df_mimiciv,  df_mimiciv_encoded],  axis=1)
df_covid_final    = pd.concat([df_covid,    df_covid_encoded],    axis=1)

df_mimiciii_final = df_mimiciii_final.drop('race', axis=1)
df_mimiciv_final  = df_mimiciv_final.drop('race', axis=1)
df_covid_final    = df_covid_final.drop('race', axis=1)

### One Hot Encoding

In [45]:
# all_races = sorted(set(df_mimiciii['race']).union(set(df_mimiciv['race'])).union(set(df_eicu['race'])))
# encoder = OneHotEncoder(categories=[all_races], sparse=False)

# encoded_mimiciii = encoder.fit_transform(df_mimiciii[['race']])
# encoded_mimiciv  = encoder.transform(df_mimiciv[['race']])
# encoded_eicu = encoder.transform(df_eicu[['race']])

# encoded_mimiciii_df = pd.DataFrame(encoded_mimiciii, columns=encoder.categories_[0])
# encoded_mimiciv_df  = pd.DataFrame(encoded_mimiciv,  columns=encoder.categories_[0])
# encoded_eicu_df = pd.DataFrame(encoded_eicu, columns=encoder.categories_[0])

# encoded_mimiciii_df = encoded_mimiciii_df.add_prefix('race_')
# encoded_mimiciv_df  = encoded_mimiciv_df.add_prefix('race_')
# encoded_eicu_df = encoded_eicu_df.add_prefix('race_')

# df_mimiciii_final = pd.concat([df_mimiciii, encoded_mimiciii_df], axis=1)
# df_mimiciv_final  = pd.concat([df_mimiciv, encoded_mimiciv_df], axis=1)
# df_eicu_final = pd.concat([df_eicu, encoded_eicu_df], axis=1)

# df_mimiciii_final = df_mimiciii_final.drop('race', axis=1)
# df_mimiciv_final  = df_mimiciv_final.drop('race', axis=1)
# df_eicu_final = df_eicu_final.drop('race', axis=1)

### Outlier Detection

In [52]:
df_mimiciii_final.loc[df_mimiciii_final['Height'] < 70, 'Height'] = 70
df_mimiciv_final.loc[df_mimiciv_final['Height'] < 70, 'Height'] = 70
df_covid_final.loc[df_covid_final['Height'] < 70, 'Height'] = 70

df_mimiciii_final.loc[df_mimiciii_final['Arterial Blood Pressure systolic'] < 10, 'Arterial Blood Pressure systolic'] = 10
df_mimiciii_final.loc[df_mimiciii_final['Arterial Blood Pressure systolic'] > 350, 'Arterial Blood Pressure systolic'] = 350

df_mimiciv_final.loc[df_mimiciv_final['MCHC'] > 42, 'MCHC'] = 42
df_covid_final.loc[df_covid_final['MCHC'] > 42, 'MCHC'] = 42

df_mimiciii_final.loc[df_mimiciii_final['Differential-Monos'] > 65, 'Differential-Monos'] = 65
df_mimiciv_final.loc[df_mimiciv_final['Differential-Monos'] > 65, 'Differential-Monos'] = 65
df_covid_final.loc[df_covid_final['Differential-Monos'] > 65, 'Differential-Monos'] = 65

df_mimiciii_final.loc[df_mimiciii_final['Triglyceride'] > 1350, 'Triglyceride'] = 1350
df_mimiciv_final.loc[df_mimiciv_final['Triglyceride'] > 1350, 'Triglyceride'] = 1350
df_covid_final.loc[df_covid_final['Triglyceride'] > 1350, 'Triglyceride'] = 1350

df_mimiciii_final.loc[df_mimiciii_final['PaO2/FiO2'] > 100, 'PaO2/FiO2'] = 100
df_mimiciv_final.loc[df_mimiciv_final['PaO2/FiO2'] > 100, 'PaO2/FiO2'] = 100
df_covid_final.loc[df_covid_final['PaO2/FiO2'] > 100, 'PaO2/FiO2'] = 100

df_mimiciii_final.loc[df_mimiciii_final['Shock_Index'] < 0, 'Shock_Index'] = 0

### Concatenate MIMIC Data

In [53]:
df_mimic = [df_mimiciii_final, df_mimiciv_final]
df_mimic_final = pd.concat(df_mimic)

In [54]:
df_mimic_final.head()

In [55]:
df_covid_final.head()

### Save Data

In [56]:
mimic_file_path = os.path.join(path_data_covid, 'mimic.parquet')
mimic_table = pa.Table.from_pandas(df_mimic_final)
pq.write_table(mimic_table, mimic_file_path)

In [57]:
covid_file_path = os.path.join(path_data_covid, 'covid.parquet')
covid_table = pa.Table.from_pandas(df_covid_final)
pq.write_table(covid_table, covid_file_path)